# Classical Machine Learning Baseline for Patient Risk Prediction

# ----------------------------
# 0. Import packages here
# ----------------------------

In [98]:
# import the needed packages
import pandas as pd
from sklearn.datasets import make_classification 
# make_classification is used to generate a random n-class classification problem
# This in simple terms means that we can generate a random dataset with a specified number of classes, features, and samples.
# This is great for generating synthetic datasets to test machine learning algorithms.


from sklearn.model_selection import train_test_split
# train_test_split is used to split the dataset into training and testing sets. 
# This is important because we want to train our model on one set of data and test it on another set of data 
# We do this to evaluate model performance on data that was not seen by the model during training.

from sklearn.tree import DecisionTreeClassifier
# DecisionTreeClassifier is a machine learning algorithm that can be used for both classification and regression tasks.
# It works by iteratively breaking the dataset into smaller pieces based on relationships between the features and the target variable.

from sklearn.ensemble import RandomForestClassifier
# Random Forest models work like democracy applied to Decision Trees. 
# Each tree in the forest makes a prediction, and the class with the most votes in the end becomes the model's prediction.
# This leads it to be referred to as "ensemble learning" where multiple models are used to make a prediction.

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# classification_report is used to evaluate the performance of a classification model by providing metrics such as precision, recall, and F1-score for each class.
# confusion_matrix is used to evaluate the performance of a classification model by providing a matrix that shows the number of true positives, true negatives, false positives, and false negatives for each class.
# accuracy_score is used to evaluate the performance by providing the overall accuracy of the model, which is the number of correct predictions divided by the total number of predictions.

import matplotlib.pyplot as plt
# Matplotlib is a data visualization library that provides a wide range of plotting capabilities.

import seaborn as sns
# Seaborn is a data visualization library based on matplotlib that provides very nice graphical representations of data. 

# ----------------------------
# 1. Create synthetic patient-like data
# ----------------------------

In [99]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

data_path = PROJECT_ROOT / "data" / "titanic" / "train.csv"
train_df = pd.read_csv(data_path)
train_df.head()

# Repeat for the test data
test_data_path = PROJECT_ROOT / "data" / "titanic" / "test.csv"
test_df = pd.read_csv(test_data_path)

# df, feature_names, classes = create_dummy_patient_data()
# df.head()
def feature_names_and_classes(df):
    feature_names = df.columns.drop("Survived").tolist()  # Create a list of feature names by dropping the target column
    classes = df["Survived"].unique().tolist()  # Identify the unique classes in the target column
    return feature_names, classes

def keep_relevant_features(df, irrelevant_features=None):
    # Keep only relevant features for the Titanic dataset
    return df.drop(columns=irrelevant_features)



# Keep only relevant features for the Titanic dataset
train_df=keep_relevant_features(train_df, irrelevant_features=(["Name","Ticket","Cabin"]))
test_df=keep_relevant_features(test_df, irrelevant_features=(["Name","Ticket","Cabin"]))

feature_names, classes = feature_names_and_classes(train_df)


In [100]:
train_df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.2500,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.9250,S
3,4,1,1,female,35.0,1,0,53.1000,S
4,5,0,3,male,35.0,0,0,8.0500,S


In [101]:
# X,y= make_classification(
#     n_samples=20000, # This is the number of samples we want to generate for the dummy data.
#     n_features=12, # This is the number of features we want to generate for the dummy data.
#     n_informative=6, # This is the number of "informative features" we want to generate for the dummy data. 
#     # Informative features are basically those that are actually useful for predicting the target variable.
#     n_redundant=2, # This is the number of redundant features we want to generate for the dummy data. 
#     # Redundant features are those that are highly correlated with other features, and hence hold no predictive value.
#     n_classes=5, # This is the number of "classes" aka unique groups we want in the dummy data.
#     random_state=42 # This is the random state for reproducibility, i.e. if you run this code multiple times, you will get the same results each time.
# )



# ----------------------------
# 2. Re-manage & Split data
# ----------------------------

In [102]:
def fill_missing_values(df):
    # Fill missing values in the "Age" column with the median age
    df["Age"] = df["Age"].fillna(df["Age"].median())

    # Fill missing values in the "Embarked" column with the most common embarkation point
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    # Convert categorical columns to numerical using one-hot encoding
    df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
    return df

train_df = fill_missing_values(train_df)
test_df = fill_missing_values(test_df)

In [103]:
X_train = train_df.drop(columns=["Survived"])
y_train = train_df["Survived"]

X_test = test_df


In [104]:
# X = df.drop(columns=["disease_risk"]) # Overwriting the X variable to only contain the features of the dataset.
# y = df["disease_risk"] # Overwriting the y variable to only contain the target variable of the dataset.


# #Split the data into train and testing sets.
# X_train, X_test, y_train, y_test = train_test_split(X,y,
#     test_size=0.2, # This is the proportion of the dataset that we want to use for testing.
#     random_state=42, # This is the random state for reproducibility.
#     stratify=y # This is used to ensure that the distribution of classes in the training and testing sets is similar to that of the original dataset.
#     # This ensures that the model is trained on a representative sample distribution.
# )

# ----------------------------
# 3. Train Decision Tree
# ----------------------------

In [105]:
dt_model = DecisionTreeClassifier(
    max_depth=10, # This is the maximum depth of the decision tree.
    random_state=42 # This is the random state for reproducibility.
    # NOTE: The documentation contains a bunch of other hyperparameters that we can tune.
)

dt_model.fit(X_train, y_train) # This is where the model is actually trained on the training data.

DecisionTreeClassifier(max_depth=10, random_state=42)

In [106]:
X_test

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,892,3,34.5,0,0,7.8292,True,True,False
1,893,3,47.0,1,0,7.0000,False,False,True
2,894,2,62.0,0,0,9.6875,True,True,False
3,895,3,27.0,0,0,8.6625,True,False,True
4,896,3,22.0,1,1,12.2875,False,False,True
...,...,...,...,...,...,...,...,...,...
413,1305,3,27.0,0,0,8.0500,True,False,True
414,1306,1,39.0,0,0,108.9000,False,False,False
415,1307,3,38.5,0,0,7.2500,True,False,True
416,1308,3,27.0,0,0,8.0500,True,False,True


In [107]:
dt_predictions = dt_model.predict(X_test) # This is where the model is used to make predictions on the testing data.
dt_results= pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": dt_predictions
})
# dt_accuracy = accuracy_score(y_test, dt_predictions) # This is where the accuracy of the model is calculated.

In [108]:
dt_results

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [109]:
results_folder = PROJECT_ROOT / "results" / "titanic"

results_folder.mkdir(parents=True, exist_ok=True)

dt_results.to_csv(results_folder / "dt_predictions.csv", index=False)

# ----------------------------
# 4. Train Random Forest
# ----------------------------

In [110]:
rf_model = RandomForestClassifier(
    n_estimators=200, # This is the number of trees in the forest. More trees usually lead to better performance, but also increase computation time.
    max_depth=8, # This is the maximum depth of each tree. Deeper trees can capture more complex patterns but may lead to overfitting.
    random_state=42 # This ensures reproducibility of the results.
)

rf_model.fit(X_train, y_train) # This is where the model is actually trained on the training data.

RandomForestClassifier(max_depth=8, n_estimators=200, random_state=42)

In [111]:
rf_predictions = rf_model.predict(X_test) # This is where the model is used to make predictions on the testing data.
rf_results= pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": rf_predictions
})


In [112]:
rf_results

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [113]:

rf_results.to_csv(results_folder / "rf_predictions.csv", index=False) # Save the predictions to a CSV file for submission to Kaggle or further analysis.


# Comparison

In [114]:
comparison= pd.DataFrame({
"PassengerId": test_df["PassengerId"],
"DecisionTree": dt_predictions,
"RandomForest": rf_predictions
})

In [115]:
# See Differences
differences= comparison[comparison["DecisionTree"] != comparison["RandomForest"]]
len(differences)

57